# Panel Interpretation: Classification Patterns and Decision Tables

This notebook provides practical guidance for interpreting panel profiles in evaluation contexts.

**Reference:** Based on evaluation scenario from blackboards/2.md (S02 Study cycle)  
**Date:** 2026-02-15  
**Purpose:** Guide evaluators in reading (Δ, S, E) profiles and making classification decisions

## Overview

The multi-component panel (Δ, S, E) uniquely characterizes researcher archetypes. This notebook demonstrates:

1. **Pattern taxonomy:** Four common patterns and their interpretations
2. **Classification thresholds:** Proposed cutoffs for evaluation decisions
3. **Decision tables:** How to translate panel values into evaluation actions
4. **Edge cases and failure modes:** When panel discrimination fails and how to mitigate
5. **Committee composition rules:** Deriving evaluation panels from researcher profiles

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Category system
categories = ['C1', 'C2', 'C3', 'C4', 'C5']
category_labels = [
    'Physics, condensed matter',
    'Materials science',
    'Chemistry, physical',
    'Biology, molecular',
    'Mathematics, applied'
]

## 1. Pattern Taxonomy: Four Common Profiles

Based on the toy examples and evaluation scenario, we identify four recurring patterns:

### Pattern 1: High Δ, High S, High E → Genuine Integrator
- **Characteristics:** Broad reference base + coherent publications + cross-field impact
- **Example:** Researcher A (toy), Dr. A (evaluation scenario)
- **Interpretation:** Cross-disciplinary research with genuine integration
- **Evaluation action:** Assess as interdisciplinary; compose cross-disciplinary committee

### Pattern 2: High Δ, Low S, Low E → Polymath (Non-Integrative)
- **Characteristics:** Broad reference base + incoherent publications + no cross-field impact
- **Example:** Researcher B (toy), Dr. B (evaluation scenario)
- **Interpretation:** Breadth without integration; multiple disciplinary contributions
- **Evaluation action:** Do NOT evaluate as interdisciplinary; split evaluation or assign to strongest field

### Pattern 3: Low Δ, High S, Low E → Specialist (Misclassified)
- **Characteristics:** Narrow reference base + coherent publications + limited cross-field impact
- **Example:** Researcher C (toy), Dr. D (evaluation scenario)
- **Interpretation:** Disciplinary specialist, possibly misclassified
- **Evaluation action:** Reclassify to disciplinary evaluation

### Pattern 4: Low Δ, Low S, Any E → Emergent/Insufficient Data
- **Characteristics:** Small publication count; unreliable signals
- **Interpretation:** Early-career researcher or insufficient data
- **Evaluation action:** Defer evaluation or use qualitative assessment

In [ ]:
# Example panel values for each pattern
patterns = pd.DataFrame({
    'Pattern': [
        '1. Genuine integrator',
        '2. Polymath (non-integrative)',
        '3. Specialist (misclassified)',
        '4. Emergent/insufficient data'
    ],
    'Δ': [0.558, 0.562, 0.288, 0.150],
    'S': [0.733, 0.000, 0.881, 0.120],
    'E': [0.600, 0.063, 0.211, 0.080],
    'Example': ['Dr. A', 'Dr. B', 'Dr. D', '(early-career)'],
    'Evaluation action': [
        'Assess as IDR; cross-disciplinary committee',
        'Split evaluation or assign to strongest field',
        'Reclassify to disciplinary evaluation',
        'Defer or use qualitative assessment'
    ]
})

print("\n" + "="*100)
print("PATTERN TAXONOMY")
print("="*100)
print(patterns.to_string(index=False))
print("="*100)

## 2. Classification Thresholds

Proposed decision thresholds for evaluation agencies:

| Classification | Δ | S | E |
|----------------|---------|---------|----------|
| Genuine integrator | ≥ 0.40 | ≥ 0.30 | ≥ 0.30 |
| Polymath (non-integrative) | ≥ 0.40 | < 0.15 | < 0.15 |
| Specialist (reclassify) | < 0.35 | any | any |
| Ambiguous (requires panel review) | else | else | else |

**IMPORTANT:** These thresholds are illustrative. Evaluation agencies must calibrate to local context based on:
- Disciplinary norms in their region
- Granularity of category system
- Policy goals (e.g., encourage cross-disciplinary vs. recognize breadth)

In [ ]:
def classify_researcher(Delta, S, E, verbose=True):
    """
    Classify researcher based on panel values using proposed thresholds.
    
    Returns: (classification, evaluation_action)
    """
    # Thresholds
    DELTA_HIGH = 0.40
    DELTA_LOW = 0.35
    S_HIGH = 0.30
    S_LOW = 0.15
    E_HIGH = 0.30
    E_LOW = 0.15
    
    # Decision logic
    if Delta >= DELTA_HIGH and S >= S_HIGH and E >= E_HIGH:
        classification = "Genuine integrator"
        action = "Assess as interdisciplinary; compose cross-disciplinary committee"
    
    elif Delta >= DELTA_HIGH and S < S_LOW and E < E_LOW:
        classification = "Polymath (non-integrative)"
        action = "Do NOT assess as IDR; split evaluation or assign to strongest field"
    
    elif Delta < DELTA_LOW:
        classification = "Specialist (reclassify)"
        action = "Reclassify to disciplinary evaluation"
    
    else:
        classification = "Ambiguous (requires panel review)"
        action = "Convene expert panel for qualitative assessment"
    
    if verbose:
        print(f"\nPanel: (Δ={Delta:.3f}, S={S:.3f}, E={E:.3f})")
        print(f"Classification: {classification}")
        print(f"Action: {action}")
    
    return classification, action

# Test on example researchers
print("\n" + "="*80)
print("CLASSIFICATION EXAMPLES")
print("="*80)

print("\n--- Dr. A (from evaluation scenario) ---")
classify_researcher(0.558, 0.733, 0.600)

print("\n--- Dr. B (from evaluation scenario) ---")
classify_researcher(0.562, 0.000, 0.063)

print("\n--- Dr. D (from evaluation scenario) ---")
classify_researcher(0.288, 0.881, 0.211)

print("\n--- Hypothetical early-career researcher ---")
classify_researcher(0.380, 0.450, 0.200)

print("\n" + "="*80)

## 3. Visual Decision Space

Visualize the classification regions in (Δ, S) space (with E as color):

In [ ]:
# Create decision space plot
fig, ax = plt.subplots(figsize=(10, 8))

# Draw threshold lines
DELTA_HIGH = 0.40
DELTA_LOW = 0.35
S_HIGH = 0.30
S_LOW = 0.15

# Regions
# Specialist: Δ < 0.35
ax.add_patch(Rectangle((0, 0), DELTA_LOW, 1.0, 
                       facecolor='lightblue', alpha=0.3, label='Specialist (reclassify)'))

# Ambiguous: 0.35 ≤ Δ < 0.40
ax.add_patch(Rectangle((DELTA_LOW, 0), DELTA_HIGH-DELTA_LOW, 1.0, 
                       facecolor='lightyellow', alpha=0.3, label='Ambiguous'))

# Genuine integrator: Δ ≥ 0.40, S ≥ 0.30
ax.add_patch(Rectangle((DELTA_HIGH, S_HIGH), 1.0-DELTA_HIGH, 1.0-S_HIGH, 
                       facecolor='lightgreen', alpha=0.3, label='Genuine integrator'))

# Polymath: Δ ≥ 0.40, S < 0.15
ax.add_patch(Rectangle((DELTA_HIGH, 0), 1.0-DELTA_HIGH, S_LOW, 
                       facecolor='lightcoral', alpha=0.3, label='Polymath (non-integrative)'))

# Plot threshold lines
ax.axvline(DELTA_LOW, color='blue', linestyle='--', linewidth=1.5, alpha=0.7)
ax.axvline(DELTA_HIGH, color='green', linestyle='--', linewidth=1.5, alpha=0.7)
ax.axhline(S_LOW, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax.axhline(S_HIGH, color='green', linestyle='--', linewidth=1.5, alpha=0.7)

# Plot example researchers
examples = [
    (0.558, 0.733, 0.600, 'Dr. A\n(integrator)'),
    (0.562, 0.000, 0.063, 'Dr. B\n(polymath)'),
    (0.288, 0.881, 0.211, 'Dr. D\n(specialist)'),
]

for Delta, S, E, label in examples:
    # Color by E value
    color = plt.cm.viridis(E)
    ax.scatter(Delta, S, s=200, c=[color], edgecolors='black', linewidth=2, zorder=10)
    ax.annotate(label, (Delta, S), xytext=(10, 10), textcoords='offset points',
               fontsize=9, bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Formatting
ax.set_xlabel('Diversity (Δ)', fontsize=12)
ax.set_ylabel('Coherence (S)', fontsize=12)
ax.set_title('Classification Decision Space (color = Cross-field effect E)', fontsize=13)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=9)

# Add colorbar for E
sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('Cross-field effect (E)', fontsize=11)

plt.tight_layout()
plt.show()

print("\nDecision space interpretation:")
print("- Blue region (Δ < 0.35): Specialists → reclassify to disciplinary evaluation")
print("- Green region (Δ ≥ 0.40, S ≥ 0.30): Genuine integrators → IDR assessment")
print("- Red region (Δ ≥ 0.40, S < 0.15): Polymaths → split or assign to strongest field")
print("- Yellow region (0.35 ≤ Δ < 0.40): Ambiguous → requires panel review")
print("- Point color indicates E (cross-field effect): darker = higher E")

## 4. Committee Composition Rule

Given a researcher classified as "genuine integrator," how should the evaluation committee be composed?

### Formal Procedure

Given researcher r with category proportion vector p_r:

1. **Identify primary categories:** $K_r = \{i : p_{r,i} \geq \tau\}$ where τ = 0.15 (threshold)
2. **Include disciplinary experts:** For each category $i \in K_r$, include ≥ 1 evaluator from discipline i
3. **Include cross-disciplinary chair:** Include ≥ 1 evaluator with demonstrated cross-disciplinary expertise (Δ > 0.4)
4. **Committee size:** $|K_r| + 1$ (disciplines + IDR chair)

### Rationale

- Disciplinary experts provide content expertise in relevant fields
- Cross-disciplinary chair understands integration mechanisms and can mediate between disciplines
- Committee spans the researcher's knowledge base without being unwieldy

In [ ]:
def compose_committee(p_vector, researcher_name="", tau=0.15):
    """
    Determine evaluation committee composition from category proportions.
    
    p_vector: category proportion vector [p1, p2, ..., p5]
    tau: threshold for "primary category" (default 0.15)
    
    Returns: (primary_categories, committee_size, committee_composition)
    """
    # Identify primary categories
    primary_indices = [i for i, p in enumerate(p_vector) if p >= tau]
    primary_cats = [categories[i] for i in primary_indices]
    
    # Committee composition
    committee_disciplines = [category_labels[i] for i in primary_indices]
    committee_size = len(primary_cats) + 1  # disciplines + IDR chair
    
    print(f"\n{'='*80}")
    print(f"COMMITTEE COMPOSITION FOR {researcher_name}")
    print(f"{'='*80}")
    print(f"\nCategory proportions:")
    for i, (cat, p, label) in enumerate(zip(categories, p_vector, category_labels)):
        marker = "*" if p >= tau else " "
        print(f"{marker} {cat} ({label:<30}): p = {p:.3f}")
    
    print(f"\nPrimary categories (p ≥ {tau}): {primary_cats}")
    print(f"\nRecommended committee composition ({committee_size} members):")
    for i, (cat, label) in enumerate(zip(primary_cats, committee_disciplines)):
        print(f"  {i+1}. Expert in {cat}: {label}")
    print(f"  {committee_size}. Cross-disciplinary chair (Δ > 0.4)")
    print(f"{'='*80}")
    
    return primary_cats, committee_size, committee_disciplines

# Example: Dr. A (genuine integrator)
p_A = np.array([0.30, 0.10, 0.20, 0.30, 0.10])
compose_committee(p_A, "Dr. A (genuine integrator)")

# Example: Dr. B (polymath, for comparison)
p_B = np.array([0.25, 0.20, 0.20, 0.20, 0.15])
compose_committee(p_B, "Dr. B (polymath - but will NOT be assessed as IDR)")

# Example: Dr. D (specialist)
p_D = np.array([0.60, 0.25, 0.10, 0.00, 0.05])
compose_committee(p_D, "Dr. D (specialist - should be reclassified)")

## 5. Edge Cases and Failure Modes

When does panel discrimination fail, and how can we mitigate?

### Failure Mode Taxonomy (6 modes)

| # | Mode | Description | Mitigation |
|---|------|-------------|------------|
| F1 | Breadth-without-depth reward | High Δ rewarded regardless of S and E | Require S > threshold AND E > threshold for "integrator" classification |
| F2 | Non-standard publication penalty | Interdisciplinary journals have lower impact factors | Use field-normalized citation indicators; do NOT compare IF across fields |
| F3 | Incommensurable impact across fields | Citation norms differ 10× between fields (math vs biology) | Normalize E by field-specific citation baselines |
| F4 | Misclassification persistence | Specialist enters "Interdisciplinar" category and stays there | Panel-based reclassification trigger: if Δ < 0.35, recommend reclassification |
| F5 | Early-career data sparsity | Junior researchers have too few publications for reliable panel values | Minimum publication threshold (e.g., n ≥ 15); use confidence intervals below threshold |
| F6 | Gaming via strategic co-authorship | Researcher adds co-authors from distant fields to inflate Δ | Weight Δ by corresponding-author publications only; cross-check S for coherence |

In [ ]:
# Failure mode examples and mitigations
failure_modes = pd.DataFrame({
    'Mode': ['F1', 'F2', 'F3', 'F4', 'F5', 'F6'],
    'Description': [
        'Breadth-without-depth reward',
        'Non-standard publication penalty',
        'Incommensurable impact across fields',
        'Misclassification persistence',
        'Early-career data sparsity',
        'Gaming via strategic co-authorship'
    ],
    'Risk': ['High', 'Medium', 'High', 'Medium', 'High', 'Medium'],
    'Mitigation': [
        'Require S ≥ threshold AND E ≥ threshold',
        'Use field-normalized citation indicators',
        'Normalize E by field-specific baselines',
        'Automatic reclassification if Δ < 0.35',
        'Minimum n ≥ 15 publications; use CIs',
        'Weight by corresponding-author; check S'
    ]
})

print("\n" + "="*100)
print("FAILURE MODE TAXONOMY")
print("="*100)
print(failure_modes.to_string(index=False))
print("="*100)

print("\nMost critical mitigations:")
print("\n1. F1 (Breadth-without-depth): This is exactly what the panel prevents!")
print("   - Type B (polymath) has high Δ but S=0 and E=0.063")
print("   - Requiring S ≥ 0.30 AND E ≥ 0.30 excludes polymaths from IDR assessment")

print("\n2. F5 (Early-career data sparsity): Fundamental statistical issue")
print("   - Panel values unreliable with n < 15 publications")
print("   - Solution: Minimum threshold OR qualitative assessment for junior researchers")

print("\n3. F6 (Gaming): Strategic behavior to inflate Δ")
print("   - Adding co-authors from distant fields increases diversity")
print("   - BUT coherence S cross-checks: if publications are incoherent, S will be low")
print("   - Panel is more robust to gaming than single-scalar Δ")
print("="*100)

## 6. When Panel Discrimination Fails

The panel approach has limits. It cannot discriminate in these situations:

1. **Early-career researchers (n < 15 publications):** Insufficient data for stable estimates
2. **Single-publication assessments:** Panel requires aggregate over multiple works
3. **Highly collaborative fields:** Hard to assign primary category when co-authorship is norm
4. **Emerging fields:** Similarity matrix may not capture new disciplinary boundaries

**Recommendation:** In these cases, fall back to qualitative expert assessment rather than forcing quantitative classification.

In [ ]:
def check_applicability(n_pubs, researcher_type, field_characteristics):
    """
    Check whether panel approach is applicable for a given researcher.
    
    n_pubs: number of publications
    researcher_type: 'individual', 'group', 'project'
    field_characteristics: dict with keys 'emerging', 'highly_collaborative'
    
    Returns: (applicable, warnings)
    """
    warnings = []
    applicable = True
    
    # Check publication count
    if n_pubs < 15:
        warnings.append(f"WARNING: n={n_pubs} < 15 (early-career or data sparsity)")
        warnings.append("  → Panel values may be unreliable; use confidence intervals or defer evaluation")
        applicable = False
    
    # Check researcher type
    if researcher_type == 'group':
        warnings.append("WARNING: Group-level assessment (not individual)")
        warnings.append("  → Panel may conflate multiple individual profiles; interpret with caution")
    
    # Check field characteristics
    if field_characteristics.get('emerging', False):
        warnings.append("WARNING: Emerging field")
        warnings.append("  → Similarity matrix may not capture new disciplinary boundaries")
        warnings.append("  → Consider updating category system before applying panel")
    
    if field_characteristics.get('highly_collaborative', False):
        warnings.append("WARNING: Highly collaborative field")
        warnings.append("  → Primary category assignment may be ambiguous")
        warnings.append("  → Consider weighting by corresponding-author publications")
    
    return applicable, warnings

# Test cases
print("\n" + "="*80)
print("APPLICABILITY CHECKS")
print("="*80)

print("\n--- Case 1: Established researcher, standard field ---")
applicable, warnings = check_applicability(
    n_pubs=30,
    researcher_type='individual',
    field_characteristics={'emerging': False, 'highly_collaborative': False}
)
print(f"Applicable: {applicable}")
if warnings:
    for w in warnings:
        print(w)
else:
    print("No warnings - panel approach fully applicable")

print("\n--- Case 2: Early-career researcher ---")
applicable, warnings = check_applicability(
    n_pubs=8,
    researcher_type='individual',
    field_characteristics={'emerging': False, 'highly_collaborative': False}
)
print(f"Applicable: {applicable}")
for w in warnings:
    print(w)

print("\n--- Case 3: Emerging field (e.g., quantum computing) ---")
applicable, warnings = check_applicability(
    n_pubs=25,
    researcher_type='individual',
    field_characteristics={'emerging': True, 'highly_collaborative': False}
)
print(f"Applicable: {applicable}")
for w in warnings:
    print(w)

print("\n--- Case 4: Highly collaborative field (e.g., particle physics) ---")
applicable, warnings = check_applicability(
    n_pubs=40,
    researcher_type='individual',
    field_characteristics={'emerging': False, 'highly_collaborative': True}
)
print(f"Applicable: {applicable}")
for w in warnings:
    print(w)

print("\n" + "="*80)

## 7. Summary Decision Table for Evaluators

Quick reference guide for evaluation agencies:

In [ ]:
# Comprehensive decision table
decision_table = pd.DataFrame({
    'Δ range': ['≥ 0.40', '≥ 0.40', '< 0.35', '0.35-0.40'],
    'S range': ['≥ 0.30', '< 0.15', 'any', 'any'],
    'E range': ['≥ 0.30', '< 0.15', 'any', 'any'],
    'Classification': [
        'Genuine integrator',
        'Polymath (non-integrative)',
        'Specialist (misclassified)',
        'Ambiguous'
    ],
    'Evaluation track': [
        'Interdisciplinary',
        'Disciplinary (strongest field)',
        'Disciplinary (reclassify)',
        'Expert panel review'
    ],
    'Committee size': [
        '|K_r| + 1',
        'N/A (split evaluation)',
        '2-3 (standard)',
        'Case-dependent'
    ]
})

print("\n" + "="*120)
print("EVALUATOR DECISION TABLE")
print("="*120)
print(decision_table.to_string(index=False))
print("="*120)

print("\nUsage instructions:")
print("1. Compute panel (Δ, S, E) for researcher")
print("2. Locate matching row in decision table")
print("3. Apply evaluation track and committee composition from table")
print("4. If n < 15 publications, override with qualitative assessment")
print("5. If ambiguous, convene expert panel for case-by-case review")

print("\nCalibration notes:")
print("- Thresholds (0.40, 0.35, 0.30, 0.15) are ILLUSTRATIVE")
print("- Agencies must calibrate to local context and policy goals")
print("- Validation: Test on known cases before deployment")
print("- Update: Revise thresholds based on stakeholder feedback")
print("="*120)

## Conclusion

This notebook provides practical interpretation guidance for the multi-component panel (Δ, S, E):

### Key Takeaways

1. **Four common patterns** emerge from panel profiles:
   - Genuine integrator: (high, high, high)
   - Polymath: (high, low, low)
   - Specialist: (low, high, low)
   - Insufficient data: (low, low, any)

2. **Classification thresholds** enable systematic evaluation:
   - Genuine integrator: Δ ≥ 0.40, S ≥ 0.30, E ≥ 0.30
   - Polymath: Δ ≥ 0.40, S < 0.15, E < 0.15
   - Specialist: Δ < 0.35 (any S, E)
   - Thresholds must be calibrated to local context

3. **Committee composition rule** derives evaluation panels from researcher profiles:
   - Include disciplinary experts for all primary categories (p ≥ 0.15)
   - Include cross-disciplinary chair (Δ > 0.4)
   - Committee size: |K_r| + 1

4. **Failure modes** have known mitigations:
   - F1 (Breadth-without-depth): Require S and E thresholds
   - F5 (Data sparsity): Minimum n ≥ 15 publications
   - F6 (Gaming): Cross-check S for coherence

5. **Edge cases** require fallback to qualitative assessment:
   - Early-career (n < 15)
   - Emerging fields
   - Highly collaborative fields

### Implementation Roadmap

For evaluation agencies:
1. Validate panel approach on known cases
2. Calibrate thresholds to local context
3. Train evaluators on pattern recognition
4. Deploy with fallback to expert review for ambiguous cases
5. Collect feedback and iterate